# Deep EDA: Viabilidad de Propagación en UNSW-NB15
## Evaluación de Grafos y Modelos SIR/SEIR

Este análisis profundiza en la estructura de red del dataset para determinar si es viable aplicar modelos epidemiológicos de propagación de amenazas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import sys
from pathlib import Path

# Agregar src al path para importar el cargador
sys.path.append('../src')
from data.unsw_loader import UNSWLoader

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
print("✓ Librerías cargadas")

## 1. Carga de Datos Consolidada
Cargamos una muestra significativa para analizar la conectividad global.

In [ ]:
loader = UNSWLoader('../data/raw/unsw-nb15')
df = loader.load_consolidated_sample(nrows_per_file=250000)

# Limpieza básica de categorías
df['attack_cat'] = df['attack_cat'].str.strip().fillna('Normal')
df.loc[df['Label'] == 0, 'attack_cat'] = 'Normal'

print(f"\n✓ Dataset cargado: {df.shape[0]:,} flujos de red")

## 2. Análisis de Estructura
Revisamos las columnas clave para el grafo (IPs) y el modelo (Labels).

In [ ]:
print("Columnas clave para el proyecto:")
key_cols = ['srcip', 'dstip', 'proto', 'service', 'attack_cat', 'Label']
display(df[key_cols].head())

## 3. Construcción del Grafo de Conectividad
Mapeamos todos los flujos de comunicación como aristas en un grafo.

In [ ]:
print("Construyendo grafo...")
G = nx.from_pandas_edgelist(df, source='srcip', target='dstip', 
                            edge_attr=['proto', 'service', 'attack_cat', 'Label'], 
                            create_using=nx.DiGraph())

print(f"✓ Grafo construido")
print(f"  - Nodos (IPs): {G.number_of_nodes()}")
print(f"  - Aristas (Comunicaciones): {G.number_of_edges()}")

## 4. Métricas de Viabilidad de Red
Calculamos métricas para evaluar si la topología permite la propagación.

In [ ]:
density = nx.density(G)
avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()

print(f"Densidad del grafo: {density:.6f}")
print(f"Grado promedio (conexiones por IP): {avg_degree:.2f}")

# Componentes conectados (usando grafo no dirigido para alcance total)
U = G.to_undirected()
components = list(nx.connected_components(U))
print(f"Número de subredes aisladas: {len(components)}")
print(f"Tamaño de la subred más grande (LCC): {len(max(components, key=len))}")

## 5. Análisis de Alcance de Amenazas (Worms/Backdoors)
¿Cuántos nodos puede 'tocar' un Gusano basándose en la topología real?

### Justificación del Modelo Epidemiológico (SIR/SEIR)
Se han seleccionado específicamente las categorías de **Worms** (Gusanos) y **Backdoors** para el modelado SIR/SEIR por las siguientes razones:

1. **Naturaleza Viral**: Los Worms están diseñados para autoreplicarse y propagarse de forma autónoma a través de las conexiones de red, lo que coincide directamente con la tasa de infección ($\beta$) en modelos epidemiológicos.
2. **Estados de Transición**: El comportamiento de estas amenazas permite mapear claramente los estados:
   - **S (Susceptible)**: Nodos sanos en la topología analizada.
   - **E (Expuesto)**: Nodos que han recibido flujos maliciosos (latencia), ideal para el modelo SEIR.
   - **I (Infectado)**: Nodos que activamente intentan propagar el ataque a sus vecinos en el grafo.
   - **R (Recuperado)**: Nodos que han sido aislados o parcheados tras detectar la amenaza.
3. **Dependencia de la Topología**: A diferencia de ataques puntuales (como DoS), la propagación viral depende de la estructura de red (quién habla con quién), validando el análisis de grafos realizado anteriormente.

In [ ]:
viral_categories = ['Worms', 'Backdoors']
infectious_flows = df[df['attack_cat'].isin(viral_categories)]
infectors = infectious_flows['srcip'].unique()

print(f"Nodos identificados como fuentes de infección: {len(infectors)}")

# Calcular alcance desde un infector promedio
if len(infectors) > 0:
    sample_infector = infectors[0]
    reachable = nx.descendants(G, sample_infector)
    print(f"\nAnálisis para Infector de Ejemplo ({sample_infector}):")
    print(f"  - Alcance directo e indirecto: {len(reachable)} nodos")
    print(f"  - % de la red que puede infectar: {(len(reachable)/G.number_of_nodes())*100:.2f}%")

## 6. Análisis Estadístico Descriptivo
Calculamos métricas clave (media, moda, etc.) para entender la naturaleza del tráfico y los ataques.

### Glosario de Variables Analizadas
Para entender las métricas estadísticas, definimos las variables clave extraídas del dataset UNSW-NB15:

*   **`dur`**: Duración total del flujo en segundos.
*   **`Spkts` / `Dpkts`**: Número total de paquetes enviados por el Origen (Source) y el Destino (Destination).
*   **`sbytes` / `dbytes`**: Volumen total de datos en bytes enviados por el Origen y el Destino.
*   **`Sload` / `Dload`**: Tasa de transferencia (bits por segundo) del Origen y del Destino.
*   **`smeansz` / `dmeansz`**: Tamaño promedio de los paquetes enviados por el Origen y el Destino (en bytes).

In [ ]:
num_cols = ['dur', 'Spkts', 'Dpkts', 'sbytes', 'dbytes', 'Sload', 'Dload', 'smeansz', 'dmeansz']

print("Métricas Estadísticas Globales (Tráfico Consolidado):")
stats_global = df[num_cols].describe().T
stats_global['mode'] = df[num_cols].mode().iloc[0]
display(stats_global[['mean', '50%', 'mode', 'std', 'min', 'max']])

print("\nDistribución de Categorías de Ataque:")
display(df['attack_cat'].value_counts())

print("\n✓ Estadísticas calculadas")

### 6.1 Comparativa por Categoría de Ataque
Analizamos cómo varían los tamaños de paquetes y volumen de datos según el tipo de amenaza.

In [ ]:
print("Promedio de métricas por categoría de ataque:")
grouped_stats = df.groupby('attack_cat')[num_cols].mean()
display(grouped_stats.sort_values(by='sbytes', ascending=False))

print("\nMediana de métricas por categoría de ataque:")
display(df.groupby('attack_cat')[num_cols].median().sort_values(by='sbytes', ascending=False))

### 6.2 Visualización de Distribuciones Estadísticas
Usamos Boxplots para comparar la dispersión del tamaño de paquetes entre categorías.

In [ ]:
plt.figure(figsize=(16, 12))

plt.subplot(2, 1, 1)
sns.boxplot(data=df, x='attack_cat', y='smeansz', palette='viridis')
plt.yscale('log')
plt.title('Distribución de Tamaño Medio de Paquetes (Origen) por Ataque', fontsize=14)
plt.xticks(rotation=45)

plt.subplot(2, 1, 2)
sns.boxplot(data=df, x='attack_cat', y='Spkts', palette='magma')
plt.yscale('log')
plt.title('Distribución de Cantidad de Paquetes (Origen) por Ataque', fontsize=14)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 7. Visualización del Grafo de Ataque
Visualizamos el subgrafo de infecciones virales.

In [ ]:
# Crear subgrafo solo con flujos de ataque viral y sus vecinos
relevant_nodes = set(infectors)
for infector in infectors:
    relevant_nodes.update(G.neighbors(infector))

S = G.subgraph(relevant_nodes)

plt.figure(figsize=(15, 10))
pos = nx.spring_layout(S, k=0.3)

node_colors = ['red' if n in infectors else 'lightblue' for n in S.nodes()]
nx.draw_networkx_nodes(S, pos, node_color=node_colors, node_size=500, alpha=0.8)
nx.draw_networkx_edges(S, pos, alpha=0.2, arrows=True)
nx.draw_networkx_labels(S, pos, font_size=8)

plt.title("Subgrafo de Amenazas Virales (Rojo = Infectores)", fontsize=15)
plt.axis('off')
plt.show()

## 8. Conclusión de Viabilidad

Basado en las métricas:
- **Conectividad:** Se han detectado componentes interconectados que permiten la propagación.
- **Viabilidad:** El modelo SIR/SEIR es aplicable debido a la existencia de rutas entre infectores y objetivos susceptibles.